# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step workflow for loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema and available at:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset main info
print(f"{metadata.name}: {metadata.description}")
print(f"Croissant schema version: {metadata.conformsTo}")
print(f"Dataset published: {metadata.datePublished}")
print(f"Keywords: {metadata.keywords}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

- All entities (record sets, fields, columns) are referenced by their `@id`.
- We'll display details for each record set, including their fields and columns.

In [ ]:
# List all available record sets by @id
record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSet]
print("Record Sets (@id):")
for rid in record_set_ids:
    print(f"- {rid}")

# For each record set, list fields and columns
for rs in dataset.metadata.recordSet:
    print(f"\nRecordSet: {rs['@id']} - {rs.get('name','N/A')}")
    fields = rs.get('field', [])
    for field in fields:
        print(f"  Field @id: {field['@id']} ({field.get('name','')})")
        columns = field.get('column', [])
        if columns:
            for column in columns:
                print(f"    Column @id: {column['@id']} ({column.get('name','')})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

- Use the record set and field `@id`s from the overview.
- All references should use `@id` for consistency.
- We'll load all available record sets into DataFrames.

In [ ]:
# Prepare list of record set @ids
record_sets = [rs['@id'] for rs in dataset.metadata.recordSet]
dataframes = {}

# Load records for each record set
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nLoaded DataFrame for {record_set_id}:")
    print(df.columns.tolist())
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps:
- Filtering records based on numeric fields (e.g., GAD-7 score or PHQ-9 score).
- Normalizing numeric fields.
- Grouping by a key demographic field.

Below, select a numeric field and group field by their `@id`. (You may need to adjust based on actual columns in the dataset.)

In [ ]:
# Choose record set and fields for EDA (adjust to match actual @ids from previous section!)
# Example field @ids (should be replaced with actual @ids from the overview)
example_record_set_id = record_sets[0] if record_sets else None
df = dataframes.get(example_record_set_id, pd.DataFrame())

# Display available columns
print("Available columns for EDA:")
print(df.columns.tolist())

# Set up fields by @id (update for your field @ids)
# Example: GAD-7 score field '@id', Gender as group field '@id'
numeric_field_id = None
group_field_id = None
# Find numeric and group fields
for col in df.columns:
    if 'score' in col.lower() or 'gad_7' in col.lower() or 'phq_9' in col.lower():
        numeric_field_id = col
    if 'gender' in col.lower():
        group_field_id = col

if numeric_field_id:
    # Filter records with score > threshold
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].head()])

    # Group by demographic field (e.g., gender)
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA. Please check available columns.")

## 5. Visualization
Visualize distributions and relationships between fields in the dataset.
- Example: Histogram of GAD-7 scores, Bar chart of mean scores by gender.

In [ ]:
# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and not df.empty:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,5))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Mental Health Survey Dataset provides a rich set of metrics on psychological symptoms among residents of Kilifi County, Kenya.
- We explored the dataset schema and loaded available record sets, referencing fields and columns by their `@id`.
- Basic EDA and visualization was performed, illustrating how you can filter, normalize, and group data using `mlcroissant`.

**Next Steps**:
- Explore relationships between other demographic and assessment fields.
- Develop predictive or clustering models using this survey data.
- Apply the workflow to other Croissant-structured datasets for reproducible analysis.